In [1]:
import pandas as pd

## Step 1: Load the data

The Credit Score dataset - 100,000 rows, one row per customer per month, so the same
`Customer_ID` appears several times. That panel structure turns out to matter later.

`pd.read_csv()` is enough to load it. No machine learning library is used anywhere in this
notebook: every step below is written with pandas only.

In [2]:

df = pd.read_csv('train.csv')

/var/folders/vt/v00ww5h94fn8k6fl1y67xsbh0000gn/T/ipykernel_3682/3907296332.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('train.csv')


In [3]:
df.shape

(100000, 28)

In [4]:
df.head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

## Step 2: Finding the missing values

Eight columns have missing values. `isnull().sum()` lists them.

In [6]:

missing_counts = df.isnull().sum()
missing_counts[missing_counts > 0].sort_values(ascending=False)

Monthly_Inhand_Salary      15002
Type_of_Loan               11408
Name                        9985
Credit_History_Age          9030
Num_of_Delayed_Payment      7002
Amount_invested_monthly     4479
Num_Credit_Inquiries        1965
Monthly_Balance             1200
dtype: int64

### Cleaning `Amount_invested_monthly`

I picked this column because the later steps need it to be numeric.

It was stored as `object`, not a number. That is the first thing worth checking - a numeric column
arriving as text usually means something non-numeric is hiding in it. Here it was values like
`'__10000__'`, which are not real amounts and not real `NaN` either. `isnull()` does not see them,
so counting nulls alone would have understated the problem.

Two steps:

1. `pd.to_numeric(..., errors='coerce')` - the corrupted strings become `NaN`, joining the genuine
   missing values so both are handled the same way.
2. Fill with the median, into a **new column**, leaving the original intact so the two can be
   compared.

Median rather than mean because the column is skewed and a few very large values would drag the
mean upward.

In [7]:
# 1. Convert string to numeric (corrupted values become NaN)
numeric_amount = pd.to_numeric(df['Amount_invested_monthly'], errors='coerce')
print('Missing values after conversion:', numeric_amount.isnull().sum())

# 2. Fill missing values with median -> save as a new column
median_value = numeric_amount.median()
df['Amount_invested_monthly_cleaned'] = numeric_amount.fillna(median_value)

print('Median value used:', median_value)
print('Missing values after cleaning:', df['Amount_invested_monthly_cleaned'].isnull().sum())
df[['Amount_invested_monthly', 'Amount_invested_monthly_cleaned']].head()

Missing values after conversion: 8784
Median value used: 128.95453805190283
Missing values after cleaning: 0


,Amount_invested_monthly,Amount_invested_monthly_cleaned
0,80.41529543900253,80.415295
1,118.28022162236736,118.280222
2,81.699521264648,81.699521
3,199.4580743910713,199.458074
4,41.420153086217326,41.420153


## Step 3: Z-score normalisation

`z = (x - mean) / std`, applied to the cleaned column.

The check that it worked is the output itself: the normalised column should have a mean of about 0
and a variance of about 1. If it does not, something upstream is wrong.

In [8]:
col = df['Amount_invested_monthly_cleaned']

df['Amount_invested_monthly_zscore'] = (col - col.mean()) / col.std()

print('Mean of z-score column:', df['Amount_invested_monthly_zscore'].mean())
print('Variance of z-score column:', df['Amount_invested_monthly_zscore'].var())

df[['Amount_invested_monthly_cleaned', 'Amount_invested_monthly_zscore']].head()

Mean of z-score column: 5.563549621001585e-17
Variance of z-score column: 1.0000000000003468


,Amount_invested_monthly_cleaned,Amount_invested_monthly_zscore
0,80.415295,-0.570546
1,118.280222,-0.372846
2,81.699521,-0.563841
3,199.458074,0.050997
4,41.420153,-0.774146


## Step 4: Binning into four equal-sized groups

Two different things get called binning. `pd.cut()` splits the *range* into equal-width intervals;
`pd.qcut()` splits the *data* into equal-count groups using quantiles. Equal numbers of records is
what is wanted here, so `qcut`.

The bins get plain labels. Names like Fail/Pass/Credit/Distinction would imply a meaning these
quartiles do not have.

In [9]:
bin_labels = ['Bin1', 'Bin2', 'Bin3', 'Bin4']

df['Amount_invested_monthly_bin'] = pd.qcut(
    df['Amount_invested_monthly_cleaned'],
    q=4,
    labels=bin_labels
)

print(df['Amount_invested_monthly_bin'].value_counts())
df[['Amount_invested_monthly_cleaned', 'Amount_invested_monthly_bin']].head()

Amount_invested_monthly_bin
Bin2    29392
Bin1    25000
Bin4    25000
Bin3    20608
Name: count, dtype: int64


,Amount_invested_monthly_cleaned,Amount_invested_monthly_bin
0,80.415295,Bin2
1,118.280222,Bin2
2,81.699521,Bin2
3,199.458074,Bin3
4,41.420153,Bin1


The bins did not come out equal: 25,000 / 29,392 / 20,608 / 25,000.

That is worth chasing rather than shrugging at, and the cause is upstream. Step 2 filled 8,784
missing values with the same median. Those 8,784 identical values all sit on one quantile boundary,
and `qcut` cannot split a tie across two bins, so the boundary is pushed and two of the four bins
absorb the imbalance.

**Imputing with a constant creates a spike in the distribution, and anything downstream that
depends on the distribution inherits it.** The bins are the visible symptom here; the same spike
would distort a histogram or a quantile-based outlier rule just as quietly.

## Step 5: One-hot encoding `Credit_Mix`

First, what is actually in the column.

In [10]:
df['Credit_Mix'].value_counts()

Credit_Mix
Standard    36479
Good        24337
_           20195
Bad         18989
Name: count, dtype: int64

`Credit_Mix` holds `Standard`, `Good`, `Bad` - and `'_'`, 20,195 times, across 10,477 distinct
customers. It is a placeholder, not a category. Encoded as-is it would produce a fourth dummy
column that means "value missing", which is not a credit mix.

The usual fix would be an imputer from a machine learning library, which is not available here. But
this dataset has something better: **the same customer appears in several months, and a customer's
credit mix does not change month to month.** So a customer whose value is missing in one month can
be filled from their own value in the others - `groupby('Customer_ID')` and take that customer's
mode.

This is more accurate than any column-wide fill, because it uses the customer's own data rather
than the population average. It only works because of how this dataset is shaped, which is the
point: looking at the structure first found a better answer than reaching for a standard tool.

In [11]:
# 1. Treat '_' as missing
df['Credit_Mix_cleaned'] = df['Credit_Mix'].replace('_', pd.NA)

# 2. Fill missing values with the same customer's mode from other months
df['Credit_Mix_cleaned'] = df.groupby('Customer_ID')['Credit_Mix_cleaned'] \
                              .transform(lambda s: s.fillna(s.mode().iloc[0] if not s.mode().empty else pd.NA))

print('Placeholder count before:', (df['Credit_Mix'] == '_').sum())
print('Missing values remaining after cleaning:', df['Credit_Mix_cleaned'].isnull().sum())
df[['Customer_ID', 'Credit_Mix', 'Credit_Mix_cleaned']].head()

Placeholder count before: 20195
Missing values remaining after cleaning: 0


,Customer_ID,Credit_Mix,Credit_Mix_cleaned
0,CUS_0xd40,_,Good
1,CUS_0xd40,Good,Good
2,CUS_0xd40,Good,Good
3,CUS_0xd40,Good,Good
4,CUS_0xd40,Good,Good


In [12]:
# 3. One-hot-encode the cleaned column and append to the DataFrame
credit_mix_onehot = pd.get_dummies(df['Credit_Mix_cleaned'], prefix='Credit_Mix')

df = pd.concat([df, credit_mix_onehot], axis=1)

df[['Credit_Mix', 'Credit_Mix_cleaned'] + list(credit_mix_onehot.columns)].head()

,Credit_Mix,Credit_Mix_cleaned,Credit_Mix_Bad,Credit_Mix_Good,Credit_Mix_Standard
0,_,Good,False,True,False
1,Good,Good,False,True,False
2,Good,Good,False,True,False
3,Good,Good,False,True,False
4,Good,Good,False,True,False
